# 02. Macro KPI Analysis & Channel ROI Performance
**Portfolio Project: Multi-Channel Marketing Analytics & Attribution Framework**  
**Company:** Grito Labs  

> **Transparency Note (Synthetic Dataset):**  
> Synthetic data generated to mimic realistic EdTech multi-channel marketing behavior (seed 42). Metrics are calibrated to credible benchmarks: channel ROAS 1.5x–8x for paid channels, email 10x–20x, CAC $20–$350, spend 18% of revenue.

---

### Business Objective
Evaluate macroeconomic acquisition efficiency across all 7 marketing channels:
- **Paid Search** (Intent capture)
- **Paid Social** (Discovery & brand consideration)
- **Display Ads** (Programmatic retargeting & awareness)
- **Email Marketing** (Owned lead nurturing)
- **Referral** (Student advocacy & affiliate partnerships)
- **Organic Search** & **Direct** (Unpaid earned equity)

Key Formulas Evaluated:
- $\text{CTR} = \frac{\text{Clicks}}{\text{Impressions}} \times 100$
- $\text{CPC} = \frac{\text{Spend}}{\text{Clicks}}$
- $\text{CPA / CAC} = \frac{\text{Spend}}{\text{Conversions}}$
- $\text{ROAS} = \frac{\text{Attributed Revenue}}{\text{Spend}}$
- $\text{ROI} = \frac{\text{Attributed Revenue} - \text{Spend}}{\text{Spend}} \times 100$



In [1]:
import sqlite3

conn = sqlite3.connect("../data/processed/marketing_analytics.db")
cursor = conn.cursor()

kpi_query = '''
WITH channel_spend AS (
    SELECT 
        s.channel_id,
        c.channel_name,
        c.is_paid,
        SUM(s.impressions) AS total_impressions,
        SUM(s.clicks) AS total_clicks,
        SUM(s.spend) AS total_spend
    FROM fact_daily_spend s
    JOIN dim_channel c ON s.channel_id = c.channel_id
    GROUP BY s.channel_id, c.channel_name, c.is_paid
),
channel_convs AS (
    SELECT 
        tp.channel_id,
        COUNT(DISTINCT cv.conversion_id) AS last_touch_conversions,
        SUM(cv.revenue) AS last_touch_revenue
    FROM fact_conversions cv
    JOIN fact_touchpoints tp ON cv.touchpoint_id = tp.touchpoint_id
    WHERE cv.conversion_cycle = 1
    GROUP BY tp.channel_id
)
SELECT 
    s.channel_name,
    s.total_impressions,
    s.total_clicks,
    ROUND(s.total_spend, 2) AS spend,
    COALESCE(c.last_touch_conversions, 0) AS conversions,
    ROUND(COALESCE(c.last_touch_revenue, 0), 2) AS revenue,
    ROUND(CAST(s.total_clicks AS REAL) / NULLIF(s.total_impressions, 0) * 100.0, 2) AS ctr_pct,
    ROUND(s.total_spend / NULLIF(s.total_clicks, 0), 2) AS cpc,
    CASE WHEN s.is_paid = 1 THEN CAST(ROUND(s.total_spend / NULLIF(c.last_touch_conversions, 0), 2) AS TEXT) ELSE 'n/a' END AS cac,
    CASE WHEN s.is_paid = 1 THEN CAST(ROUND(COALESCE(c.last_touch_revenue, 0) / NULLIF(s.total_spend, 0), 2) AS TEXT) ELSE 'n/a' END AS last_touch_roas
FROM channel_spend s
LEFT JOIN channel_convs c ON s.channel_id = c.channel_id
ORDER BY s.total_spend DESC;
'''
cursor.execute(kpi_query)
print("=== CHANNEL PERFORMANCE & UNIT ECONOMICS ===")
print(f"{'Channel':16} | {'Spend':>11} | {'Conv':>5} | {'Revenue':>11} | {'CTR %':>5} | {'CPC':>5} | {'CAC':>7} | {'ROAS':>6}")
print("-" * 85)
for r in cursor.fetchall():
    ch, imp, clk, sp, cnv, rev, ctr, cpc, cac, roas = r
    print(f"{ch:16} | ${sp:>10,.2f} | {cnv:>5} | ${rev:>10,.2f} | {ctr:>5.2f}% | ${cpc:>4.2f} | ${cac:>6} | {roas:>6}")



=== CHANNEL PERFORMANCE & UNIT ECONOMICS ===
Channel          |       Spend |  Conv |     Revenue | CTR % |   CPC |     CAC |   ROAS
-------------------------------------------------------------------------------------
Paid Search      | $198,410.22 |   505 | $506,125.30 |  4.24% | $9.54 | $392.89 |   2.55
Paid Social      | $142,951.06 |   408 | $387,351.40 |  1.10% | $5.51 | $350.37 |   2.71
Display Ads      |  $51,626.47 |   206 | $198,153.30 |  0.29% | $1.93 | $250.61 |   3.84
Referral         |  $19,729.96 |   103 | $100,378.90 | 66.83% | $2.77 | $191.55 |   5.09
Email Marketing  |  $13,700.33 |   358 | $333,974.30 |  2.81% | $0.78 |  $38.27 |  24.38
Organic Search   |       $0.00 |   307 | $302,134.50 | 66.88% | $0.00 |     $n/a |    n/a
Direct           |       $0.00 |   563 | $554,396.90 | 66.80% | $0.00 |     $n/a |    n/a


### 💡 'So What?' — Strategic Interpretation
1. **Email Marketing is an Ultra-High-Margin Engine**: With a Last-Touch ROAS of **24.4x** and CAC of **$38.27**, Email converts warm leads with minimal marginal expense.
2. **Paid Search Absorbs 46.5% of Budget**: Paid Search captures high purchase intent ($506K revenue), but delivers a lower baseline ROAS (2.55x) with CAC ($392.89).
3. **Display Ads & Paid Social Discrepancy**: On a Last-Touch basis, Paid Social delivers 2.71x ROAS, but multi-touch analysis reveals it originates hundreds of thousands in pipeline that converts via Direct.

